# GRPO ile Çoktan Seçmeli Sorularda Tutarlılık Analizi

Bu notebook, OpenBookQA tabanlı çoktan seçmeli soru cevaplama deneylerinde kullanılan eğitim ve değerlendirme akışını **yönlendirici** biçimde sunar.

Çalışmanın ana fikri şudur:

- Önce teacher modelden açıklamalı SFT verisi üretilir.
- Qwen3.5-4B student model SFT ile göreve hazırlanır.
- Ardından farklı GRPO tabanlı stratejiler karşılaştırılır:
  - **mA_v1:** Standart outcome reward GRPO
  - **mA_v2:** Hiyerarşik outcome reward GRPO
  - **mB_v1:** Orijinal + benzer soru tutarlılığı
  - **mB_v2:** Güçlü çeldiricilerle ablasyon
  - **mC:** DAPO tarzı dynamic sampling / Clip-Higher ablasyonu
  - **mD:** Process supervision + Gaussian step reward + DAPO

> Not: Bu notebook tek seferde baştan sona çalıştırılmak zorunda değildir. Bazı hücreler pahalı teacher generation veya uzun eğitim içerir. Her bölümün başındaki açıklamayı okuyarak yalnızca ihtiyacınız olan hücreleri çalıştırın.


## Deney Akışı

Genel akış iki ana parçadan oluşur: **veri hazırlama** ve **model karşılaştırması**.

```text
ARC + OpenBookQA
        ↓
SFT Pool
        ↓
Teacher Rationales
        ↓
Final SFT Dataset
        ↓
Qwen3.5-4B SFT Checkpoint
        ↓
────────────────────────────────────────
        ↓                ↓
mA: Standard GRPO     mB: Pair Consistency GRPO
        ↓                ↓
mC: DAPO Ablation     mD: Process Supervision + DAPO
        ↓                ↓
Evaluation on original + similar OpenBookQA test pairs
```

### Çalıştırma modları

| Mod | Ne zaman kullanılır? | Çalıştırılacak ana bölümler |
|---|---|---|
| İlk kurulum | Veri ve checkpoint yoksa | Adım 1 → Adım 6 |
| Sadece GRPO eğitimi | SFT checkpoint hazırsa | Adım 7 → Adım 10 |
| Sadece değerlendirme | Modeller eğitildiyse | Adım 9 |
| Ablasyon | mB sonrası ek deneyler için | Adım 10 |


## Donanım ve Model Varsayımları

Bu çalışma Google Colab üzerinde yüksek bellekli GPU ortamı hedeflenerek hazırlanmıştır.

| Bileşen | Kullanılan model / araç |
|---|---|
| Teacher model | Qwen2.5-72B-Instruct-AWQ |
| Student model | Qwen3.5-4B |
| Eğitim framework'ü | Transformers + TRL |
| Önerilen GPU | NVIDIA RTX PRO A6000 GPU |



## Adım 1 — Ortam Kurulumu

Bu bölüm Colab ortamını hazırlar:

1. Google Drive bağlanır.
2. Proje Drive'dan `/content/project_grpo` dizinine kopyalanır.
3. Gerekli paketler kurulur.
4. GPU ve dosya yolları kontrol edilir.

> Kendi Drive yapınız farklıysa `PROJECT_DRIVE` değişkenini güncelleyin.


In [ ]:
# Drive bağlantısı ve gerekli kütüphanelerin kurulumu
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DRIVE = "/content/drive/MyDrive/project_grpo"
PROJECT_LOCAL = "/content/project_grpo"

!rsync -av {PROJECT_DRIVE}/ {PROJECT_LOCAL}/
%cd /content/project_grpo



In [ ]:
!pip install -r requirements.txt

In [ ]:
# Versiyon kontrolü
import torch

print("Torch:", torch.__version__)
print("CUDA aktif mi?:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Dosyaların kontrolü
!ls /content/project_grpo/scripts
!ls /content/project_grpo/data/processed/splits

## Adım 2 — Veri Setlerinin İndirilmesi

Bu hücre ARC ve OpenBookQA veri setlerini indirir.


Veriler daha önce indirildiyse bu adımı geçebilirsiniz.


In [ ]:
!python scripts/01_download_datasets.py

## Adım 3 — Veri Bölme

Bu adım SFT, GRPO ve test kümelerini oluşturur.

Beklenen temel çıktılar:

- `sft_train_pool.jsonl`
- `openbookqa_grpo_train_1000.jsonl`
- `openbookqa_test_500.jsonl`

Bu dosyalar zaten varsa bu hücreyi tekrar çalıştırmak zorunlu değildir.


In [ ]:
!python scripts/02_create_splits.py

## Adım 4 — Teacher Rationale Üretimi

Bu adımda teacher model kullanılarak SFT için açıklamalı cevaplar üretilir.

**Amaç:** Student modelin yalnızca doğru şıkkı değil, kısa gerekçelendirme formatını da öğrenmesini sağlamak.



In [ ]:
# Qwen2.5-72B-Instruct-AWQ modelinin çalışması için gerekli kütüphane
!pip install gptqmodel

In [ ]:
!python scripts/04_generate_sft_rationales.py \
  --input_file /content/project_grpo/data/processed/splits/sft_train_pool.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/data/teacher_outputs/sft_rationales_train.jsonl \
  --model_name Qwen/Qwen2.5-72B-Instruct-AWQ \
  --max_new_tokens 160 \
  --temperature 0.2 \
  --top_p 0.9 \
  --resume

### SFT Veri Setini Oluşturma

Teacher çıktıları filtrelenerek nihai SFT veri seti oluşturulur.

Bu aşamada amaç:

- geçerli formatta olmayan örnekleri ayırmak,
- doğru cevap / açıklama yapısını korumak,
- student model için temiz bir SFT veri seti üretmektir.



In [ ]:
!python scripts/05_build_sft_dataset.py \
  --input_file {PROJECT_DRIVE}/data/teacher_outputs/sft_rationales_train.jsonl \
  --output_file {PROJECT_DRIVE}/data/processed/final/sft_train.jsonl \
  --invalid_file {PROJECT_DRIVE}/data/processed/final/sft_invalid.jsonl

In [ ]:
!wc -l /content/drive/MyDrive/project_grpo/data/processed/final/sft_train.jsonl
!wc -l /content/drive/MyDrive/project_grpo/data/processed/final/sft_invalid.jsonl
!cat /content/drive/MyDrive/project_grpo/data/processed/final/sft_train.stats.json

## Adım 5 — Student Modelin SFT Eğitimi

Bu adımda Qwen3.5-4B modeli teacher tarafından oluşturulan açıklamalı veriyle SFT edilir.

SFT checkpoint daha sonra bütün GRPO varyantlarının başlangıç modeli olarak kullanılır.

Beklenen çıktı:

```text
/content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full
```

> Eğer bu checkpoint hazırsa bu eğitim hücresini atlayabilirsiniz.


In [ ]:
!python scripts/06_train_sft.py \
  --model_name Qwen/Qwen3.5-4B \
  --train_file /content/drive/MyDrive/project_grpo/data/processed/final/sft_train.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --training_mode full \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --max_seq_length 1024 \
  --learning_rate 1e-5 \
  --save_steps 250 \
  --logging_steps 10 \
  --gradient_checkpointing

## Adım 6 — Benzer Soru Üretimi

mB, mC ve mD deneyleri için her OpenBookQA sorusuna teacher model tarafından anlamsal olarak benzer ikinci bir soru üretilir.

Bu bölüm iki çıktı üretir:

- Eğitim için benzer soru çiftleri
- Test için benzer soru çiftleri

> Bu adım da teacher model kullandığı için maliyetlidir. Çıktı dosyaları hazırsa tekrar çalıştırmayın.


In [ ]:
!pip install gptqmodel

> `gptqmodel` AWQ teacher model kullanımı için gerekebilir. Ortamınızda zaten kuruluysa bu hücreyi tekrar çalıştırmanız gerekmez.


In [ ]:
# OpenbookQA 1000 benzer eğitim sorusu üretimi
!python scripts/08_generate_similar_questions.py \
  --input_file /content/project_grpo/data/processed/splits/openbookqa_grpo_train_1000.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/data/teacher_outputs/openbookqa_grpo_train_1000_similar.jsonl \
  --model_name Qwen/Qwen2.5-72B-Instruct-AWQ \
  --resume

### Test Seti için Benzer Soru Üretimi

Eğitim seti için benzer sorular üretildikten sonra aynı işlem test seti için de yapılır. Değerlendirmede original/similar accuracy ve pair consistency bu dosyaya dayanır.


In [ ]:
# OpenbookQA 500 benzer test sorusu üretimi
!python scripts/08_generate_similar_questions.py \
  --input_file /content/project_grpo/data/processed/splits/openbookqa_test_500.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/data/teacher_outputs/openbookqa_test_500_similar.jsonl \
  --model_name Qwen/Qwen2.5-72B-Instruct-AWQ \
  --resume

## 6a. Paired Dataset Oluşturma ve Şık Karıştırma

Bu adımda teacher model tarafından üretilen similar question dosyaları, orijinal OpenBookQA dosyalarıyla `id` üzerinden eşleştirilir.

Bu işlem sonucunda mB, mC ve mD modellerinin beklediği çift sorulu format oluşturulur:

- `question_1`, `choices_1`, `answer_1`
- `question_2`, `choices_2`, `answer_2`

Ayrıca similar question üretimi sırasında başarısız olan örnekler filtrelenir. Bu yüzden train dosyası 1000 örnekten 993 örneğe, test dosyası ise 500 örnekten 493 örneğe düşer.

Son olarak Soru 2'nin şıkları karıştırılır. Böylece modelin iki soruda aynı cevap harfine bakarak kestirme öğrenmesi engellenir.

In [ ]:
!python scripts/06a_build_paired_datasets.py \
  --original_train data/processed/splits/openbookqa_grpo_train_1000.jsonl \
  --similar_train data/teacher_outputs/openbookqa_grpo_train_1000_similar.jsonl \
  --output_train data/processed/final/openbook_train_paired_shuffled.jsonl \
  --original_test data/processed/splits/openbookqa_test_500.jsonl \
  --similar_test data/teacher_outputs/openbookqa_test_500_similar.jsonl \
  --output_test data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --seed 42

## 7a. Paired Veriden mA İçin Tekli Veri Çıkarma

mA modeli tek soru üzerinden eğitildiği/değerlendirildiği için paired dosyalardaki yalnızca orijinal soru kısmı alınır.

Bu adımda:

- `openbook_train_paired_shuffled.jsonl` → `openbook_train_993.jsonl`
- `openbookqa_test_paired_shuffled.jsonl` → `openbook_test_493.jsonl`

dosyaları oluşturulur.

Böylece mA ile mB/mC/mD karşılaştırması aynı örnek tabanı üzerinde yapılır.

In [ ]:
!python scripts/07a_extract_single_from_pairs.py \
  --input_train data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_train data/processed/final/openbook_train_993.jsonl \
  --input_test data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_test data/processed/final/openbook_test_493.jsonl

## Adım 7 — Standart GRPO Deneyi: mA

mA, temel karşılaştırma modelidir.

Bu deneyde model yalnızca nihai cevabın doğru olup olmamasına göre ödül alır. Böylece diğer yöntemlerin, özellikle benzer soru tutarlılığı ve process supervision katkısının karşılaştırılabileceği bir baseline elde edilir.

> Notebook'taki komut `grpo_mA_v2` çıktısını üretir. Bu çalıştırma ayarları mA için kullanılan deneysel ayarlardır.


In [ ]:
# mA modeli standart GRPO Eğitimi
!python scripts/07_train_grpo_mA.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --train_file /content/project_grpo/data/processed/final/openbook_train_993.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mA_v2 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 350 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --logging_steps 5


## Adım 8 — Similar Question GRPO Deneyi: mB

mB deneyinde modelden aynı anda iki soruyu cevaplaması beklenir:

1. Orijinal OpenBookQA sorusu
2. Teacher tarafından üretilen anlamsal olarak benzer soru

Amaç, modelin yalnızca tek soruda doğru şıkkı bulmasını değil, aynı bilgiye dayanan benzer soruda da kararlı cevap vermesini teşvik etmektir.


In [ ]:
# mB modeli benzer soru ödüllü GRPO eğitimi
!python /content/project_grpo/scripts/09_train_grpo_mB_1.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --train_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mB_1 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 500 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --beta 0.1 \
  --logging_steps 5

## Adım 9 — Değerlendirme

Bu bölüm SFT, mA ve mB modellerini aynı test seti üzerinde değerlendirir.

Kullanılan test dosyası:

```text
/content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl
```

`--mode single` tek cevap üreten modeller için, `--mode dual` ise orijinal + benzer soru cevabı üreten modeller için kullanılır.


## Değerlendirme Metrikleri

| Metrik | Anlamı |
|---|---|
| Original Accuracy | Orijinal OpenBookQA test sorularındaki doğruluk |
| Similar Accuracy | Teacher tarafından üretilen benzer sorulardaki doğruluk |
| Both Correct / Pair Consistency | Orijinal ve benzer sorunun birlikte doğru cevaplanma oranı |

Bu metrikler birlikte okunmalıdır. Örneğin yalnızca original accuracy yüksek ama similar accuracy düşükse modelin kararlı genelleme yapmadığı düşünülebilir.


In [ ]:
# SFT modeli değerlendirmesi
# Original Accuracy + Similar Accuracy + Pair Consistency
!python /content/project_grpo/scripts/10_evaluate_models.py \
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file  /content/drive/MyDrive/project_grpo/outputs/evaluation/sft_eval_paired.json \
  --mode single \
  --max_new_tokens 350

In [ ]:
# mA modeli değerlendirmesi
# Original Accuracy + Similar Accuracy + Pair Consistency
!python /content/project_grpo/scripts/10_evaluate_models.py\
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/grpo_mA_v2 \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file  /content/drive/MyDrive/project_grpo/outputs/evaluation/mA_v2_eval_paired.json \
  --mode single \
  --max_new_tokens 350

In [ ]:
# mB modeli değerlendirmesi
# Original Accuracy + Similar Accuracy + Pair Consistency
!python /content/project_grpo/scripts/10_evaluate_models.py \
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/grpo_mB_1 \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file  /content/drive/MyDrive/project_grpo/outputs/evaluation/mB_1_v2_eval_paired.json \
  --mode dual \
  --max_new_tokens 700

# Adım 10 — Ablasyon Testleri

Bu bölüm ana mA/mB karşılaştırmasından sonra yapılan ek deneyleri içerir.

Ablasyonların amacı:

- güçlü çeldiricilerin etkisini ölçmek,
- DAPO tarzı dynamic sampling ve Clip-Higher katkısını incelemek,
- process supervision + Gaussian step reward stratejisinin genelleme davranışına etkisini test etmektir.

Bu hücreleri **yalnızca ilgili scriptler ve veri dosyaları hazırsa** çalıştırın.


In [ ]:
!pip install gptqmodel

## Ablasyon 10.1 — Güçlü Çeldirici Üretimi

Bu bölümde mevcut orijinal + benzer soru çiftleri korunur, ancak seçenekler daha güçlü çeldiricilerle yeniden düzenlenir.

Önce küçük bir örnek üretmek için `--limit 10` ile test çalıştırması yapılır. Çıktı makul görünüyorsa tam üretime geçilir.


In [ ]:
!python /content/project_grpo/scripts/11_refine_distractors_existing_pairs.py \
  --input_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_file /content/project_grpo/data/processed/final/openbook_train_paired_strong_distractors_SAMPLE.jsonl \
  --teacher_model Qwen/Qwen2.5-72B-Instruct-AWQ \
  --student_model /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --limit 10 \
  --max_attempts 3 \
  --teacher_max_new_tokens 900 \
  --teacher_temperature 0.8 \
  --teacher_top_p 0.95 \
  --student_max_new_tokens 256 \
  --accept_if_one_wrong \
  --overwrite

### Tam Güçlü Çeldirici Üretimi

Örnek üretim doğru görünüyorsa aşağıdaki hücre tam eğitim seti için güçlü çeldirici üretir.


In [ ]:
!python /content/project_grpo/scripts/11_refine_distractors_existing_pairs.py \
  --input_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_file /content/project_grpo/data/processed/final/openbook_train_paired_strong_distractors.jsonl \
  --teacher_model Qwen/Qwen2.5-72B-Instruct-AWQ \
  --student_model /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --limit 993 \
  --max_attempts 3 \
  --teacher_max_new_tokens 900 \
  --teacher_temperature 0.8 \
  --teacher_top_p 0.95 \
  --student_max_new_tokens 256 \
  --accept_if_one_wrong \
  --overwrite

### mB_v2 Modeli Eğitimi

Bu deney mB reward yapısını korur; fark, eğitim verisindeki seçeneklerin daha güçlü çeldiriciler içermesidir.

Amaç, güçlü çeldiricilerin MCQA tabanlı RLVR ödül sinyalini nasıl etkilediğini ölçmektir.


In [ ]:
!python /content/project_grpo/scripts/09_train_grpo_mB_1.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --train_file /content/project_grpo/data/processed/final/openbook_train_paired_strong_distractors.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mB_2_strong_distractors \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 500 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --beta 0.1 \
  --logging_steps 5

### mB_v2 Değerlendirmesi

Bu değerlendirme, güçlü çeldiricilerle eğitilen mB varyantının standart paired test setindeki davranışını ölçer.


In [ ]:
# mB_v2 modeli değerlendirmesi
# Original Accuracy + Similar Accuracy + Pair Consistency
!python /content/project_grpo/scripts/10_evaluate_models.py \
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/grpo_mB_2_strong_distractors \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/outputs/evaluation/mB_2_strong_distractors_eval_paired.json \
  --mode dual \
  --max_new_tokens 700


## Ablasyon 10.2 — mC: DAPO Tarzı GRPO

mC, mB reward yapısını temel alır ve DAPO'dan esinlenen şu bileşenleri ekler:

- dynamic sampling,
- Clip-Higher,
- token-level / DAPO tarzı loss seçimi.

Bu deney, optimizasyon stratejisinin mB'ye göre ek katkı sağlayıp sağlamadığını test eder.


In [ ]:
# mC modeli: mB reward + DAPO dynamic sampling + Clip-Higher + token-level loss
!python /content/project_grpo/scripts/12_train_grpo_mC_dapo.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --train_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mC_dapo \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 500 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --beta 0.1 \
  --epsilon 0.2 \
  --epsilon_high 0.28 \
  --loss_type dapo \
  --dynamic_max_retries 3 \
  --logging_steps 5

### mC Değerlendirmesi

mC dual cevap ürettiği için değerlendirme `--mode dual` ile yapılır.


In [ ]:
# mC DAPO modeli değerlendirmesi
!python /content/project_grpo/scripts/10_evaluate_models.py \
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/grpo_mC_dapo \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/outputs/evaluation/mC_dapo_eval_paired.json \
  --mode dual \
  --max_new_tokens 700


### mD için Ek Paketler

Aşağıdaki paketler mD teacher/process reward deneylerinde tokenizer/model yükleme uyumluluğu için kullanılır.


In [ ]:
!pip install -q sentencepiece tiktoken protobuf

## Ablasyon 10.3 — mD: Process Supervision + Gaussian Step Reward

mD, outcome reward'a ek olarak reasoning sürecini de puanlamayı hedefler.

İki varyant bulunmaktadır:

1. **Teacher scoring kapalı:** Gaussian process reward yapısını teacher olmadan test eder.
2. **Teacher scoring açık:** Teacher model süreç kalitesini de puanlamaya dahil eder.



### mD Varyantı 1 — Teacher Scoring Kapalı

Bu hücre process reward mekanizmasını teacher modelin puanlaması olmadan yapar.



In [ ]:
# mD Varyantı 1: teacher scoring kapalı
!python /content/project_grpo/scripts/13_train_grpo_mD_teacher_gaussian_dapo.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --teacher_model_name_or_path Qwen/Qwen2.5-32B-Instruct-AWQ \
  --train_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mD_process_no_teacher \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 500 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --beta 0.1 \
  --epsilon 0.2 \
  --epsilon_high 0.2 \
  --loss_type grpo \
  --no_dynamic_sampling \
  --process_weight 0.2 \
  --step_outside_penalty -0.2 \
  --gaussian_sigma_divisor 3.0 \
  --disable_teacher_scoring \
  --logging_steps 5


### mD Varyantı 2 — Teacher Gaussian DAPO

Bu hücre mD'nin ana/final deney varyantını çalıştırır.

Teacher scoring, Gaussian step reward ve DAPO tarzı optimizasyon birlikte kullanılır.


In [ ]:
# mD Varyantı 2: teacher Gaussian DAPO
!python /content/project_grpo/scripts/13_train_grpo_mD_teacher_gaussian_dapo.py \
  --model_name_or_path /content/drive/MyDrive/project_grpo/outputs/models/sft_qwen35_4b_full \
  --teacher_model_name_or_path Qwen/Qwen2.5-32B-Instruct \
  --train_file /content/project_grpo/data/processed/final/openbook_train_paired_shuffled.jsonl \
  --output_dir /content/drive/MyDrive/project_grpo/outputs/models/grpo_mD_teacher_gaussian_dapo \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --num_generations 4 \
  --max_completion_length 500 \
  --temperature 0.7 \
  --top_p 0.95 \
  --learning_rate 2e-6 \
  --beta 0.1 \
  --epsilon 0.2 \
  --epsilon_high 0.28 \
  --loss_type dapo \
  --dynamic_max_retries 3 \
  --process_weight 0.2 \
  --teacher_max_new_tokens 256 \
  --teacher_load_in_4bit \
  --logging_steps 5


In [ ]:
# mD Teacher Gaussian DAPO modeli değerlendirmesi
!python /content/project_grpo/scripts/10_evaluate_models.py \
  --model_path /content/drive/MyDrive/project_grpo/outputs/models/grpo_mD_teacher_gaussian_dapo \
  --test_file /content/project_grpo/data/processed/final/openbookqa_test_paired_shuffled.jsonl \
  --output_file /content/drive/MyDrive/project_grpo/outputs/evaluation/mD_teacher_gaussian_dapo_eval_paired.json \
  --mode dual \
  --max_new_tokens 700


## GitHub'a Yüklemeden Önce Kontrol Listesi

Notebook'u paylaşmadan önce şunları kontrol edin:

- Çıktılar temizlendi mi? Büyük loglar ve model çıktıları notebook içinde kalmamalı.
- Drive'a özel yollar açıkça belirtilmiş mi?
- Pahalı hücrelerin başında “ne zaman çalıştırılmalı?” açıklaması var mı?
- Model checkpointleri GitHub'a eklenmemeli; `.gitignore` içinde `outputs/models/` gibi klasörler dışlanmalı.
- Veri dosyaları büyükse GitHub yerine Drive/Hugging Face linki verilmelidir.
- Sonuç JSON/CSV dosyaları küçükse `outputs/evaluation/` altında paylaşılabilir.

Önerilen GitHub dosya adı:

```text
project_grpo_guided.ipynb
```
